# Weight Initialization in PyTorch: Eight Layers, One Output

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/weight_initialization.ipynb)

Most PyTorch code never sets a weight initialization, so it runs on whatever `nn.Linear` does
by default. That default is neither Xavier nor He. In a plain stack of linear layers and ReLUs
it gives the network the same output for every input by about eight layers deep. At eight
layers SGD sits at chance for several epochs before it escapes; at twenty it never does.

This notebook reads the default out of PyTorch's source, predicts its behaviour from one line
of algebra, checks every prediction against a measurement, and shows the depth at which
training stalls, and the one-line fix.

Companion post: [sesen.ai/blog/weight-initialization-pytorch](https://sesen.ai/blog/weight-initialization-pytorch)

Runtime: about three minutes on a Colab CPU. No GPU needed.

In [ ]:
import inspect
import math

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets

## 1. Data and the model

10,000 FashionMNIST training images (a fixed random subset) and the full 10,000-image test set,
standardised with the subset's mean and standard deviation. The model is a plain MLP: `depth`
hidden layers of width 256, each a `nn.Linear` followed by a ReLU, then a linear output layer.

In [ ]:
tr = datasets.FashionMNIST("data", train=True, download=True)
te = datasets.FashionMNIST("data", train=False, download=True)
idx = torch.randperm(60_000, generator=torch.Generator().manual_seed(0))[:10_000]
x_tr, y_tr = tr.data[idx].float().div(255).view(-1, 784), tr.targets[idx]
x_te, y_te = te.data.float().div(255).view(-1, 784), te.targets
mu, sd = x_tr.mean(), x_tr.std()
x_tr, x_te = (x_tr - mu) / sd, (x_te - mu) / sd


def mlp(depth, width=256, batchnorm=False):
    layers, d = [], 784
    for _ in range(depth):
        layers.append(nn.Linear(d, width))
        if batchnorm:
            layers.append(nn.BatchNorm1d(width))
        layers.append(nn.ReLU())
        d = width
    return nn.Sequential(*layers, nn.Linear(d, 10))


def init_(model, scheme):
    """'default' leaves PyTorch's reset_parameters alone; 'default_nobias' keeps the
    default weights and zeroes the biases."""
    if scheme == "default":
        return
    for m in model.modules():
        if isinstance(m, nn.Linear):
            if scheme == "xavier":
                nn.init.xavier_normal_(m.weight)
            elif scheme == "he":
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            nn.init.zeros_(m.bias)


def build(depth, scheme, seed=0, batchnorm=False):
    torch.manual_seed(seed)
    model = mlp(depth, batchnorm=batchnorm)
    init_(model, scheme)
    return model


@torch.no_grad()
def input_share(model, x):
    """Fraction of the output's energy that changes from one input to the next."""
    out = model(x)
    return (out.var(dim=0).mean() / out.pow(2).mean()).item()


def train(model, epochs=3, lr=0.01, opt="sgd", seed=0):
    if opt == "adam":
        optim = torch.optim.Adam(model.parameters(), lr=1e-3)
    else:
        optim = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    g = torch.Generator().manual_seed(seed)
    for _ in range(epochs):
        model.train()
        for b in torch.randperm(10_000, generator=g).split(128):
            loss = F.cross_entropy(model(x_tr[b]), y_tr[b])
            optim.zero_grad()
            loss.backward()
            optim.step()
    model.eval()
    with torch.no_grad():
        return (model(x_te).argmax(1) == y_te).float().mean().item()

## 2. The headline

Same architecture, data, optimiser and seed. The only difference is whether He initialization
is applied. The post's numbers came from four CPU threads; on other hardware He's accuracies
move in the third decimal, while the default's 0.100 reproduces exactly.

In [ ]:
for depth in (4, 8, 20):
    for name in ("default", "he"):
        model = build(depth, name)
        share = input_share(model, x_tr[:512])
        print(f"depth {depth:2d}  {name:7s}  input share {share:.1e}  "
              f"test accuracy {train(model):.3f}")

## 3. The default, read from the source

`kaiming_uniform_` with `a=sqrt(5)` is the uniform box `[-1/sqrt(n), 1/sqrt(n)]`, whose
variance is `1/(3n)`. So `n * Var[w]` should be 1/3, against He's 2.

In [ ]:
print(inspect.getsource(nn.Linear.reset_parameters))

torch.manual_seed(0)
lin, conv = nn.Linear(256, 256), nn.Conv2d(64, 64, 3)
print("nn.Linear  n * Var[w] =", round(lin.weight.var().item() * 256, 3))
print("nn.Conv2d  n * Var[w] =", round(conv.weight.var().item() * 64 * 9, 3))
print("largest |w| =", round(lin.weight.abs().max().item(), 4), " 1/sqrt(256) =", 1 / 16)

## 4. One layer, one multiplication

Each linear layer multiplies the signal's second moment by `n * Var[w]`, and each ReLU halves
it, so the gain per layer is `g = n * Var[w] / 2`: 1/6 for the default, 1/2 for Xavier (when
fan-in equals fan-out), 1 for He.

Trace a 30-layer network at initialization and record each hidden layer's activation standard
deviation and the input-dependent share of its pre-activations.

In [ ]:
@torch.no_grad()
def layer_trace(model, x):
    stds, shares = [], []
    h = x
    for layer in model:
        h = layer(h)
        if isinstance(layer, nn.Linear):
            shares.append((h.var(dim=0).mean() / h.pow(2).mean()).item())
        elif isinstance(layer, nn.ReLU):
            stds.append(h.std().item())
    return stds, shares


colours = {"default": "#c2453a", "default_nobias": "#d99120", "xavier": "#4a6d8c", "he": "#1f9e9e"}
traces = {s: layer_trace(build(30, s), x_tr[:512]) for s in colours}

fig, ax = plt.subplots(figsize=(8, 4))
for s, (stds, _) in traces.items():
    ax.plot(range(1, 31), stds, color=colours[s], label=s)
ax.set_yscale("log")
ax.set_xlabel("hidden layer")
ax.set_ylabel("activation std")
ax.legend()
plt.show()

## 5. The bias floor

The default's activations stop falling at about 0.023. Each layer shrinks the signal by six
and adds a fresh bias with variance `1/(3n)`, so the pre-activation variance obeys
`q' = q/6 + 1/(3n)`, with fixed point `q* = 2/(5n)`. After a ReLU, a zero-mean Gaussian with
variance `q` has standard deviation `sqrt(q (pi - 1) / (2 pi))`.

In [ ]:
n = 256
q_star = 2 / (5 * n)
print("predicted floor, post-ReLU std:", round(math.sqrt(q_star * (math.pi - 1) / (2 * math.pi)), 4))
print("measured, layers 20 to 30:     ", np.round(traces["default"][0][19:], 4))

## 6. The input drains away

The bias is the same for every input. Each layer shrinks the part of the signal that came from
the image by six and adds back a part that came from nowhere, so the input-dependent share
should fall by a factor of about six per layer once the floor is reached. Zeroing the biases
removes the effect, which isolates the cause.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for s in ("he", "default_nobias", "default"):
    ax.plot(range(1, 32), traces[s][1], color=colours[s], label=s)
ax.set_yscale("log")
ax.set_xlabel("layer (31 is the output)")
ax.set_ylabel("input-dependent share")
ax.legend()
plt.show()

d = traces["default"][1]
print("per-layer ratio, layers 5 to 10:", [round(d[i] / d[i + 1], 2) for i in range(4, 9)])

## 7. The backward pass

Backpropagation multiplies the gradient's second moment by the same gain per layer, so the
default's gradient norm should shrink by about `sqrt(6) = 2.45` per layer on the way back.

In [ ]:
def grad_norms(model, x, y):
    model.zero_grad()
    F.cross_entropy(model(x), y).backward()
    return [m.weight.grad.norm().item() for m in model.modules() if isinstance(m, nn.Linear)]


fig, ax = plt.subplots(figsize=(8, 4))
for s in ("he", "xavier", "default"):
    g = grad_norms(build(20, s), x_tr[:512], y_tr[:512])
    ax.plot(range(1, 22), g, marker="o", ms=3, color=colours[s], label=s)
    if s == "default":
        print("default, per-layer ratio from layer 8 to the output:", round((g[20] / g[7]) ** (1 / 13), 2))
ax.set_yscale("log")
ax.set_xlabel("layer (21 is the output)")
ax.set_ylabel("weight-gradient norm at step 0")
ax.legend()
plt.show()

## 8. The depth at which training stalls

Three epochs, one seed per point to keep the notebook quick; the post uses three seeds. At
eight layers the default is at chance after three epochs but escapes given six to eight more;
the post shows that run.

In [ ]:
depths = [2, 4, 6, 8, 12]
acc = {s: [train(build(k, s)) for k in depths] for s in ("default", "he")}
for s, a in acc.items():
    print(s.ljust(8), [round(v, 3) for v in a])

fig, ax = plt.subplots(figsize=(6, 3.5))
for s, a in acc.items():
    ax.plot(depths, a, marker="o", color=colours[s], label=s)
ax.axhline(0.1, color="grey", ls="--", lw=0.8)
ax.set_xlabel("hidden layers")
ax.set_ylabel("test accuracy, 3 epochs SGD")
ax.legend()
plt.show()

## 9. Learning rate, Adam and batch normalisation

Three checks on the depth-20 result. No learning rate rescues the default under SGD. Adam,
which rescales each parameter's step by its own gradient size, partly rescues it, and the
amount varies from run to run. Batch normalisation after every layer makes the two
initializations indistinguishable.

In [ ]:
for lr in (0.003, 0.03, 0.1):
    print(f"SGD lr {lr}:  default {train(build(20, 'default'), lr=lr):.3f}   "
          f"he {train(build(20, 'he'), lr=lr):.3f}")

print(f"Adam:        default {train(build(20, 'default'), opt='adam'):.3f}   "
      f"he {train(build(20, 'he'), opt='adam'):.3f}")

print(f"BatchNorm:   default {train(build(20, 'default', batchnorm=True)):.3f}   "
      f"he {train(build(20, 'he', batchnorm=True)):.3f}")

## 10. Applying an initialization in practice

`model.apply` visits every submodule, so one function covers the whole network. Note that
`a` in `kaiming_*_` is a leaky ReLU's negative slope, not a scale: `a=math.sqrt(5)` gives you
PyTorch's default, not He initialization.

In [ ]:
def init_weights(m):
    if isinstance(m, (nn.Linear, nn.Conv2d)):
        nn.init.kaiming_normal_(m.weight, mode="fan_in", nonlinearity="relu")
        if m.bias is not None:
            nn.init.zeros_(m.bias)


model = mlp(20)
model.apply(init_weights)
print("input share after init_weights:", f"{input_share(model, x_tr[:512]):.2f}")
print("test accuracy:", f"{train(model):.3f}")

## Exercises

1. **Find the floor for other widths.** The predicted floor is `sqrt(2/(5n) * (pi-1)/(2pi))`.
   Measure it at widths 64 and 1024 and check that it scales as `1/sqrt(n)`.

2. **Make the default trainable without changing its weights.** Zero only the biases of a
   depth-20 default network. What happens to the input share, and to training under SGD and
   under Adam? Section 6 suggests what to expect from each.

3. **Tanh.** Swap ReLU for tanh and repeat section 4 with Xavier and with
   `xavier_normal_(w, gain=nn.init.calculate_gain("tanh"))`. Which keeps the signal's scale?

4. **The edge of chaos.** Under He with zero biases the input share still drifts down, slowly.
   Measure it at depth 100. Then add biases with variance 0.01 and repeat. Which way does the
   network move relative to the critical line `n * Var[w] / 2 = 1`?

5. **Residual connections.** Replace each hidden layer with `x + relu(linear(x))` and train at
   depth 20 under the default. The post did not measure this arm; predict the result first.

## Where this comes from

- LeCun, Bottou, Orr and Müller (1998), "Efficient BackProp"
- Glorot and Bengio (2010), "Understanding the difficulty of training deep feedforward neural networks"
- He, Zhang, Ren and Sun (2015), arXiv:1502.01852
- Schoenholz, Gilmer, Ganguli and Sohl-Dickstein (2017), "Deep Information Propagation"
- PyTorch issues #15314 and #57109, on the history of the default